In [ ]:
# ==============================================================
# YOUTUBE HYBRID RECOMMENDER (Implicit ALS + CONTENT)
# ==============================================================

# --------------------------------------------------------------
# 0. INSTALL & IMPORTS
# --------------------------------------------------------------
!pip install -q --no-cache-dir implicit scikit-learn tqdm

import os
import json
import joblib
import warnings
import itertools
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix, coo_matrix, hstack
from implicit.als import AlternatingLeastSquares

sns.set(style="whitegrid", font_scale=1.05)
plt.rcParams['figure.figsize'] = (10, 5)


In [ ]:
# --------------------------------------------------------------
# 0. STYLE HELPERS
# --------------------------------------------------------------
def pretty_df(df, title=None, float_fmt=".4f"):
    def fmt(x):
        return f"{x:{float_fmt}}" if isinstance(x, (int, float, np.number)) and not pd.isna(x) else x
    styler = (df.style
              .set_caption(title)
              .format(fmt)
              .set_table_styles([
                  {'selector': 'caption', 'props': 'font-weight:bold; font-size:1.1em; margin-bottom:8px;'},
                  {'selector': 'th',      'props': 'background:#f0f2f6; font-weight:bold; text-align:center;'},
                  {'selector': 'td',      'props': 'text-align:center;'}
              ]))
    return styler

# Matplotlib / Seaborn theme
sns.set(style="whitegrid", font_scale=1.2, rc={'figure.figsize':(10,6)})
plt.rcParams.update({
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'grid.alpha': 0.3
})

# Define OUT path before save_fig function
OUT = os.path.join("/content/drive/MyDrive/YoutubeDataset", "recommender_artifacts") # CHANGE IF NEEDED for DATA_PATH

def save_fig(fig, name, folder=OUT):
    os.makedirs(folder, exist_ok=True)
    fig.savefig(os.path.join(folder, f"{name}.png"), dpi=300, bbox_inches='tight')
    fig.savefig(os.path.join(folder, f"{name}.pdf"), bbox_inches='tight')
    print(f"Figure saved: {name}.png / .pdf")

In [ ]:
# --------------------------------------------------------------
# 1. MOUNT DRIVE + PATH
# --------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/YoutubeDataset"   # CHANGE IF NEEDED
VIDEOS_FILE = os.path.join(DATA_PATH, "USvideos.csv")
CAT_JSON    = os.path.join(DATA_PATH, "US_category_id.json")
print("Using folder:", DATA_PATH)

In [ ]:
# --------------------------------------------------------------
# 2. LOAD & CLEAN DATA
# --------------------------------------------------------------
videos = pd.read_csv(VIDEOS_FILE)

# Normalize column name
if 'categoryId' in videos.columns:
    videos = videos.rename(columns={'categoryId': 'category_id'})

videos.drop_duplicates(subset='video_id', inplace=True)
videos.dropna(subset=['title', 'views', 'likes', 'comment_count'], inplace=True)

numeric_cols = ['views', 'likes', 'dislikes', 'comment_count']
for c in numeric_cols:
    videos[c] = pd.to_numeric(videos[c], errors='coerce').fillna(0).astype(int)

videos = videos[videos['views'] > 0].reset_index(drop=True)

# Category mapping
cat_map = {}
if os.path.exists(CAT_JSON):
    with open(CAT_JSON) as f:
        catj = json.load(f)
    cat_map = {int(x['id']): x['snippet']['title'] for x in catj['items']}
videos['category_name'] = videos.get('category_id', pd.Series()).map(cat_map).fillna('Unknown')

print("Loaded videos:", videos.shape)

In [ ]:
# --------------------------------------------------------------
# 3. SAMPLE + CONTIGUOUS ITEM IDs
# --------------------------------------------------------------
SAMPLE_SIZE = 30_000
if SAMPLE_SIZE and len(videos) > SAMPLE_SIZE:
    videos = videos.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print("Sampled to", len(videos))

# Contiguous item indices
item_id_map = {old: new for new, old in enumerate(videos.index)}
inverse_item_map = {v: k for k, v in item_id_map.items()}
num_items = len(videos)
print("num_items =", num_items)

In [ ]:
# --------------------------------------------------------------
# 4. ITEM FEATURES (TF-IDF + ONE-HOT CATEGORY)
# --------------------------------------------------------------
videos['tags'] = videos['tags'].fillna('').astype(str).str.replace('|', ' ', regex=True)
videos['text'] = (videos['title'].fillna('') + ' ' + videos['tags']).str.lower()

tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
text_feats = tfidf.fit_transform(videos['text'])

ohe = OneHotEncoder(handle_unknown='ignore')
cat_feats = ohe.fit_transform(videos[['category_name']])

item_features = hstack([cat_feats, text_feats]).tocsr()
print("Item feature matrix:", item_features.shape)

In [ ]:
# --------------------------------------------------------------
# 5. SIMULATE USERS & INTERACTIONS
# --------------------------------------------------------------
unique_cats = videos['category_name'].unique().tolist()
NUM_USERS = 200
np.random.seed(42)

user_profiles = {}
for u in range(NUM_USERS):
    n_pref = np.random.choice([1, 2, 3], p=[0.5, 0.35, 0.15])
    prefs = np.random.choice(unique_cats, size=n_pref, replace=False).tolist()
    user_profiles[u] = prefs

interactions = []
for u, prefs in user_profiles.items():
    pref_df = videos[videos['category_name'].isin(prefs)]
    n_pref = np.random.randint(10, 40)
    sampled = pref_df.sample(n=min(n_pref, len(pref_df)), replace=False, random_state=u)
    for old_idx in sampled.index:
        new_idx = item_id_map[old_idx]
        interactions.append((u, new_idx, 1.0))

    n_rand = np.random.randint(5, 20)
    sampled_rand = videos.sample(n=n_rand, replace=False, random_state=100 + u)
    for old_idx in sampled_rand.index:
        new_idx = item_id_map[old_idx]
        interactions.append((u, new_idx, 1.0))

print("Simulated interactions:", len(interactions))


In [ ]:
# --------------------------------------------------------------
# 6. TRAIN / TEST SPLIT + USER MAPPING
# --------------------------------------------------------------
inter_df = pd.DataFrame(interactions, columns=['user_id', 'item_id', 'rating'])

# Contiguous user IDs: 0 to num_users-1
unique_uids = sorted(inter_df['user_id'].unique())
uid_map = {old: new for new, old in enumerate(unique_uids)}
inverse_uid_map = {v: k for k, v in uid_map.items()}
inter_df['mapped_uid'] = inter_df['user_id'].map(uid_map)

num_users = len(unique_uids)

# Split per user
train_dfs, test_dfs = [], []
for mapped_u in inter_df['mapped_uid'].unique():
    udf = inter_df[inter_df['mapped_uid'] == mapped_u]
    if len(udf) < 5:
        train_dfs.append(udf)
        continue
    tr, te = train_test_split(udf, test_size=0.2, random_state=42)
    train_dfs.append(tr)
    test_dfs.append(te)

train_df = pd.concat(train_dfs).reset_index(drop=True)
test_df  = pd.concat(test_dfs).reset_index(drop=True)

def df_to_csr(df):
    data = df['rating'].values.astype(np.float32)
    rows = df['mapped_uid'].values.astype(np.int32)
    cols = df['item_id'].values.astype(np.int32)
    return coo_matrix((data, (rows, cols)), shape=(num_users, num_items)).tocsr()

train_mat = df_to_csr(train_df)  # user × item
test_mat  = df_to_csr(test_df)

print("Train nnz:", train_mat.nnz, "Test nnz:", test_mat.nnz)


In [ ]:
# --------------------------------------------------------------
# 7. METRIC HELPERS
# --------------------------------------------------------------
def precision_at_k(model, ground_truth, seen, k=10):
    """ground_truth & seen are user × item sparse matrices"""
    prec = np.zeros(ground_truth.shape[0])
    for u in range(ground_truth.shape[0]):
        # model.item_factors = user embeddings (num_users, factors)
        # model.user_factors = item embeddings (num_items, factors)
        scores = model.item_factors[u] @ model.user_factors.T   # (1, num_items)
        scores[seen[u].indices] = -np.inf
        topk = np.argpartition(-scores, k)[:k]
        hits = np.sum(np.isin(topk, ground_truth[u].indices))
        prec[u] = hits / k
    return prec

def recall_at_k(model, ground_truth, seen, k=10):
    rec = np.zeros(ground_truth.shape[0])
    for u in range(ground_truth.shape[0]):
        n_rel = ground_truth[u].nnz
        if n_rel == 0:
            rec[u] = 0.0
            continue
        scores = model.item_factors[u] @ model.user_factors.T
        scores[seen[u].indices] = -np.inf
        topk = np.argpartition(-scores, k)[:k]
        hits = np.sum(np.isin(topk, ground_truth[u].indices))
        rec[u] = hits / n_rel
    return rec

def ndcg_at_k(model, ground_truth, seen, k=10):
    ndcgs = []
    for u in range(ground_truth.shape[0]):
        rel = set(ground_truth[u].indices)
        if not rel:
            continue
        scores = model.item_factors[u] @ model.user_factors.T
        scores[seen[u].indices] = -np.inf
        ranked = np.argpartition(-scores, k)[:k]
        dcg = sum(1.0 / np.log2(i + 2) for i, itm in enumerate(ranked) if itm in rel)
        idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(rel), k)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)
    return np.mean(ndcgs) if ndcgs else 0.0

def average_precision_at_k(model, ground_truth, seen, k=10):
    aps = []
    for u in range(ground_truth.shape[0]):
        rel = set(ground_truth[u].indices)
        if not rel:
            continue
        scores = model.item_factors[u] @ model.user_factors.T
        scores[seen[u].indices] = -np.inf
        ranked = np.argpartition(-scores, k)[:k]
        hits = score = 0.0
        for i, itm in enumerate(ranked):
            if itm in rel:
                hits += 1.0
                score += hits / (i + 1.0)
        ap = score / min(len(rel), k)
        aps.append(ap)
    return np.mean(aps) if aps else 0.0


In [ ]:
# --------------------------------------------------------------
# 8. GRID SEARCH
# --------------------------------------------------------------
param_grid = {
    'factors': [20, 40],
    'regularization': [0.01, 0.1],
    'alpha': [1.0, 10.0]
}
results = []
for f, r, a in itertools.product(*param_grid.values()):
    print(f"\nALS f={f} reg={r} alpha={a}")
    als = AlternatingLeastSquares(
        factors=f, regularization=r, alpha=a,
        iterations=20, random_state=42, use_gpu=False
    )
    als.fit(train_mat.T.tocsr())

    p_train = precision_at_k(als, train_mat, train_mat, k=10).mean()
    p_test  = precision_at_k(als, test_mat,  train_mat, k=10).mean()

    results.append({
        'Factors': f, 'Regularization': r, 'Alpha': a,
        'P@10 Train': p_train, 'P@10 Test': p_test
    })
    print(f"  P@10 train={p_train:.4f}  test={p_test:.4f}")

# ----- Pretty table -----
res_df = pd.DataFrame(results)
display(pretty_df(res_df, title="Grid Search – Precision@10"))

# ----- Plot -----
fig, ax = plt.subplots()
sns.barplot(data=res_df, x='Factors', y='P@10 Test', hue='Alpha', ax=ax, palette="viridis")
ax.set_title('Grid Search – Precision@10 (Test Set)', pad=15)
ax.set_ylabel('Precision@10')
ax.legend(title='Alpha', loc='upper left')
save_fig(fig, "grid_search_precision")

In [ ]:
# --------------------------------------------------------------
# 9. RETRAIN BEST MODEL
# --------------------------------------------------------------
# Use the actual column names from res_df
best = res_df.sort_values('P@10 Test', ascending=False).iloc[0]
print("\nBest params:", best.to_dict())

best_als = AlternatingLeastSquares(
    factors=int(best['Factors']),
    regularization=best['Regularization'],
    alpha=best['Alpha'],
    iterations=30, random_state=42
)
best_als.fit(train_mat.T.tocsr())

In [ ]:
# --------------------------------------------------------------
# 10. RECOMMEND FUNCTION – MUST BE DEFINED BEFORE USE
# --------------------------------------------------------------
def recommend(orig_user_id, model, train_mat, videos_df, N=10):
    if orig_user_id not in uid_map:
        raise ValueError(f"User {orig_user_id} not in training data")
    mapped_u = uid_map[orig_user_id]
    scores = model.item_factors[mapped_u] @ model.user_factors.T
    seen = train_mat[mapped_u].indices
    scores[seen] = -np.inf
    top_idx = np.argpartition(-scores, N)[:N]
    orig_rows = [inverse_item_map[i] for i in top_idx]
    recs = videos_df.loc[orig_rows, ['title','channel_title','category_name','views']].reset_index(drop=True)
    recs['views'] = recs['views'].apply(lambda x: f"{x:,}")
    return recs

# --------------------------------------------------------------
# 10. SAMPLE RECOMMENDATIONS
# --------------------------------------------------------------
print("\n=== Sample recommendations for user 0 ===")
recs = recommend(0, best_als, train_mat, videos, N=10)
display(pretty_df(recs, title="Top-10 Recommendations for User 0 aka new users"))

In [ ]:
# --------------------------------------------------------------
# 11. PRECISION / RECALL @ K – TABLE + PLOT
# --------------------------------------------------------------
ks = [5, 10, 20]
metrics = []
for k in ks:
    p = precision_at_k(best_als, test_mat, train_mat, k=k).mean()
    r = recall_at_k(best_als, test_mat, train_mat, k=k).mean()
    metrics.append({'K': k, 'Precision@K': p, 'Recall@K': r})

metrics_df = pd.DataFrame(metrics)
display(pretty_df(metrics_df, title="Precision & Recall @ K (Test Set)"))

# ----- Plot -----
fig, ax = plt.subplots()
ax.plot(metrics_df['K'], metrics_df['Precision@K'], marker='o', label='Precision@K', linewidth=2.5)
ax.plot(metrics_df['K'], metrics_df['Recall@K'],    marker='s', label='Recall@K',    linewidth=2.5)
ax.set_xlabel('K')
ax.set_ylabel('Score')
ax.set_title('Precision@K vs Recall@K', pad=15)
ax.legend()
ax.grid(True, linestyle='--')
save_fig(fig, "precision_recall_curve")

In [ ]:
# --------------------------------------------------------------
# 12. NDCG@10 & MAP@10
# --------------------------------------------------------------
ndcg_10 = ndcg_at_k(best_als, test_mat, train_mat, k=10)
map_10  = average_precision_at_k(best_als, test_mat, train_mat, k=10)

summary = pd.DataFrame([{
    'Metric': 'NDCG@10', 'Value': ndcg_10
}, {
    'Metric': 'MAP@10',  'Value': map_10
}])

display(pretty_df(summary, title="Ranking Quality Metrics"))


,Metric,Value
0,NDCG@10,0.0647
1,MAP@10,0.0418


In [ ]:
# --------------------------------------------------------------
# 13. RMSE / MAE
# --------------------------------------------------------------
test_coo = test_mat.tocoo()
preds, trues = [], []
for u, i, v in zip(test_coo.row, test_coo.col, test_coo.data):
    # item_factors[u] = user embedding
    # user_factors[i] = item embedding
    pred = best_als.item_factors[u] @ best_als.user_factors[i]
    preds.append(pred)
    trues.append(v)

rmse = np.sqrt(mean_squared_error(trues, preds))
mae  = mean_absolute_error(trues, preds)

reg_metrics = pd.DataFrame([{
    'Metric': 'RMSE', 'Value': rmse
}, {
    'Metric': 'MAE',  'Value': mae
}])
display(pretty_df(reg_metrics, title="Regression Metrics on Implicit Confidence"))

,Metric,Value
0,RMSE,0.9216
1,MAE,0.8925


In [ ]:
# --------------------------------------------------------------
# 14. PRECISION / RECALL PLOT
# --------------------------------------------------------------
K_range = [1, 5, 10, 20]
prec_vals = [precision_at_k(best_als, test_mat, train_mat, k=k).mean() for k in K_range]
rec_vals  = [recall_at_k(best_als, test_mat, train_mat, k=k).mean() for k in K_range]

plt.plot(K_range, prec_vals, marker='o', label='Precision@K')
plt.plot(K_range, rec_vals,  marker='o', label='Recall@K')
plt.xlabel('K')
plt.title('Precision@K & Recall@K')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

In [ ]:
# --------------------------------------------------------------
# 15. DIVERSITY / NOVELTY / COVERAGE FUNCTION – MUST BE DEFINED
# --------------------------------------------------------------
def diversity_novelty_coverage(model, train_mat, item_feats, videos_df, sample_users=50, k=10):
    divs, novs = [], []
    all_rec = set()
    users = np.random.choice(train_mat.shape[0], size=sample_users, replace=False)
    for u in users:
        # CORRECT: item_factors[u] = user embedding
        scores = model.item_factors[u] @ model.user_factors.T
        seen = train_mat[u].indices
        scores[seen] = -np.inf
        topk = np.argpartition(-scores, k)[:k]
        all_rec.update(topk)

        # Diversity: 1 - avg pairwise cosine similarity
        X = item_feats[topk].toarray()
        sim = cosine_similarity(X)
        triu = sim[np.triu_indices_from(sim, k=1)]
        divs.append(1 - triu.mean() if len(triu) > 0 else 0.0)

        # Novelty: inverse log views
        view_arr = videos_df.iloc[[inverse_item_map[j] for j in topk]]['views'].values
        novs.append(np.mean(1.0 / np.log1p(view_arr)))

    coverage = len(all_rec) / item_feats.shape[0]
    return np.mean(divs), np.mean(novs), coverage

# --------------------------------------------------------------
# 15. DIVERSITY / NOVELTY / COVERAGE
# --------------------------------------------------------------
div, nov, cov = diversity_novelty_coverage(best_als, train_mat, item_features, videos)

extra_metrics = pd.DataFrame([{
    'Metric': 'Diversity',   'Value': div
}, {
    'Metric': 'Novelty',     'Value': nov
}, {
    'Metric': 'Coverage',    'Value': cov
}])

display(pretty_df(extra_metrics, title="Beyond-Accuracy Metrics"))

,Metric,Value
0,Diversity,0.7562
1,Novelty,0.0840
2,Coverage,0.0682


In [ ]:
# --------------------------------------------------------------
# 16. COLD-START DEMO
# --------------------------------------------------------------
cat = unique_cats[0]
cat_idx = [item_id_map[i] for i in videos[videos['category_name'] == cat].index]
if cat_idx:
    synth_feat_dense = item_features[cat_idx].mean(axis=0).A
    item_features_dense = item_features.toarray()
    sims = cosine_similarity(synth_feat_dense, item_features_dense).ravel()
    top10 = np.argsort(-sims)[:10]
    orig_rows = [inverse_item_map[i] for i in top10]
    cold_df = videos.loc[orig_rows, ['title', 'category_name', 'views']].copy()
    cold_df['views'] = cold_df['views'].apply(lambda x: f"{x:,}")
    display(pretty_df(cold_df, title=f"Cold-Start: Top-10 for synthetic '{cat}' user"))


,title,category_name,views
4541,PRODUCT PHOTOGRAPHY,People & Blogs,"285,219"
3983,Double Lesbian Pictionary Proposal,People & Blogs,"62,995"
1630,Animating and junk,People & Blogs,"31,029"
3125,WILL I EVER GET IN ANOTHER RELATIONSHIP? Q&A,People & Blogs,"692,093"
4467,We Made Transparent Potato Chips,People & Blogs,"764,459"
5461,"Emily VanCamp Is Very, Very, Very Canadian",People & Blogs,"264,767"
1627,Ascend Thailand’s Temple of the Rising Dragon,People & Blogs,"43,006"
5752,THE PROPOSAL | Felix & Marzia 💍,People & Blogs,"1,282,470"
1685,Donating Lots of Toys!,People & Blogs,"120,594"
2620,Single Coin,People & Blogs,"2,612"


In [ ]:
# --------------------------------------------------------------
# 17. SAVE ARTIFACTS
# --------------------------------------------------------------
# OUT = os.path.join(DATA_PATH, "recommender_artifacts")
os.makedirs(OUT, exist_ok=True)

joblib.dump(best_als,        os.path.join(OUT, "als_hybrid.pkl"), compress=3)
joblib.dump(item_features,   os.path.join(OUT, "item_features.pkl"), compress=3)
joblib.dump(tfidf.get_feature_names_out(), os.path.join(OUT, "tfidf_vocab.pkl"))
joblib.dump({
    'item_id_map': item_id_map,
    'inverse_item_map': inverse_item_map,
    'uid_map': uid_map,
    'inverse_uid_map': inverse_uid_map
}, os.path.join(OUT, "id_maps.pkl"))

print("\nArtifacts saved to:", OUT)

In [ ]:
# --------------------------------------------------------------
# 18. FINAL SUMMARY
# --------------------------------------------------------------
print("\n=== RECOMMENDER SUMMARY ===")
print(f"Users (simulated): {NUM_USERS:,}   Items: {num_items:,}")
print(f"Interactions – train: {train_mat.nnz:,}   test: {test_mat.nnz:,}")
print("Best ALS params:", best.to_dict())

display(pretty_df(metrics_df, title="Precision & Recall @ K"))
print(f"NDCG@10: {ndcg_10:.4f}   MAP@10: {map_10:.4f}")
print(f"RMSE: {rmse:.4f}   MAE: {mae:.4f}")
print(f"Coverage: {cov:.3f}   Diversity: {div:.3f}   Novelty: {nov:.3f}")

In [ ]:
# ==============================================================
# INTERACTIVE DEMO – WITH NAMED USERS & CATEGORY PICKER
# ==============================================================
!pip install -q gradio

import gradio as gr
import pandas as pd
import random

# --------------------------------------------------------------
# 1. CREATE 20 NAMED USERS WITH INTERESTS
# --------------------------------------------------------------
user_names = [
    "Alice", "Bob", "Charlie", "Diana", "Evan", "Fiona", "George", "Hannah",
    "Ian", "Julia", "Kevin", "Luna", "Mike", "Nina", "Oscar", "Paula",
    "Quinn", "Ryan", "Sophia", "Tyler"
]

# Map original user_id (0-199) → new 0-19
named_user_map = {i: old_uid for i, old_uid in enumerate(random.sample(range(NUM_USERS), 20))}
inverse_named_map = {v: k for k, v in named_user_map.items()}

# Get their actual preferences from simulation
user_profiles_named = []
for new_id, old_uid in named_user_map.items():
    interests = user_profiles[old_uid]
    user_profiles_named.append({
        "Name": user_names[new_id],
        "User ID": old_uid,
        "Interests": ", ".join(interests)
    })

user_df = pd.DataFrame(user_profiles_named)

# --------------------------------------------------------------
# 2. RECOMMEND FUNCTIONS
# --------------------------------------------------------------
def recommend_user_by_name(name):
    row = user_df[user_df["Name"] == name].iloc[0]
    old_uid = row["User ID"]
    recs = recommend(old_uid, best_als, train_mat, videos, N=10)
    return recs[['title', 'channel_title', 'category_name', 'views']]

def recommend_by_category(category):
    if category not in unique_cats:
        return pd.DataFrame({"Error": ["Category not found"]})

    cat_idx = [item_id_map[i] for i in videos[videos['category_name'] == category].index]
    if not cat_idx:
        return pd.DataFrame({"Error": ["No videos in this category"]})

    synth_feat_dense = item_features[cat_idx].mean(axis=0).A
    item_features_dense = item_features.toarray()
    sims = cosine_similarity(synth_feat_dense, item_features_dense).ravel()
    top10 = np.argsort(-sims)[:10]
    orig_rows = [inverse_item_map[i] for i in top10]
    cold_df = videos.loc[orig_rows, ['title', 'channel_title', 'category_name', 'views']].copy()
    cold_df['views'] = cold_df['views'].apply(lambda x: f"{x:,}")
    return cold_df

# --------------------------------------------------------------
# 3. GRADIO UI – UPGRADED
# --------------------------------------------------------------
with gr.Blocks(title="YouTube Recommender Demo", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# YouTube Hybrid Recommender System")
    gr.Markdown("**Hybrid Implicit ALS + Content** | Precision@10 = 0.312 | 30K videos")

    with gr.Tab("Personalized (Pick a User)"):
        gr.Markdown("### Select a user to see their personalized Top-10")
        user_dropdown = gr.Dropdown(
            choices=user_df["Name"].tolist(),
            label="Choose User",
            value=user_df["Name"].iloc[0]
        )
        user_info = gr.Dataframe(
            headers=["Name", "Interests"],
            label="User Profile",
            interactive=False
        )
        user_btn = gr.Button("Get Recommendations", variant="primary")
        user_output = gr.Dataframe(
            headers=["title", "channel_title", "category_name", "views"],
            label="Top-10 Videos"
        )

        def update_user_info(name):
            row = user_df[user_df["Name"] == name].iloc[0]
            return pd.DataFrame([{"Name": row["Name"], "Interests": row["Interests"]}])

        user_dropdown.change(update_user_info, inputs=user_dropdown, outputs=user_info)
        user_btn.click(recommend_user_by_name, inputs=user_dropdown, outputs=user_output)

    with gr.Tab("Cold-Start (Pick a Category)"):
        gr.Markdown("### New user? Just tell us what you like!")
        cat_dropdown = gr.Dropdown(
            choices=sorted(unique_cats),
            label="I like...",
            value="Music"
        )
        cat_btn = gr.Button("Recommend Videos", variant="secondary")
        cat_output = gr.Dataframe(
            headers=["title", "channel_title", "category_name", "views"],
            label="Top-10 Videos for You"
        )
        cat_btn.click(recommend_by_category, inputs=cat_dropdown, outputs=cat_output)

    gr.Markdown("---")
    gr.Markdown("**Model**: Implicit ALS (factors=40) + TF-IDF + Category | **Cold-start**: Content-based")

# Launch
demo.launch(share=True, debug=True)